# Lab 10 - Tokenizer Alignment & Sliding Window (`tokenizer_alignment.ipynb`)
Kiểm tra chi tiết cơ chế xử lý đầu vào của `AutoTokenizer` cho bài toán Extractive Question Answering với `truncation="only_second"`, `stride=128`, `return_overflowing_tokens=True`, và `return_offsets_mapping=True`.

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

question = "What is the main mechanism used by Transformers?"
# Tạo context dài (> 384 tokens) để kích hoạt cơ chế sliding window
context = (
    "The Transformer architecture, introduced in 2017, revolutionized Natural Language Processing. "
    "Unlike recurrent neural networks, it relies entirely on the self-attention mechanism to draw global dependencies between input and output. "
) * 25

inputs = tokenizer(
    question,
    context,
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length"
)

print("Các khóa (keys) trả về:", list(inputs.keys()))
print("Số lượng chunk (cửa sổ trượt) tạo ra:", len(inputs["input_ids"]))
print("-" * 60)

# 1. Kiểm tra input_ids
print("1. input_ids (15 token đầu của chunk 0):", inputs["input_ids"][0][:15])
print("   Giải mã lại text:", tokenizer.decode(inputs["input_ids"][0][:15]))

# 2. Kiểm tra attention_mask
print("\n2. attention_mask (15 giá trị đầu của chunk 0):", inputs["attention_mask"][0][:15])
print("   Tổng token thật (mask=1) ở chunk 0:", sum(inputs["attention_mask"][0]))
print("   Tổng token thật (mask=1) ở chunk cuối:", sum(inputs["attention_mask"][-1]))

# 3. Kiểm tra offset_mapping
print("\n3. offset_mapping (15 cặp đầu của chunk 0):", inputs["offset_mapping"][0][:15])
print("   Sequence IDs (15 giá trị đầu):", inputs.sequence_ids(0)[:15])

# 4. Kiểm tra overflow_to_sample_mapping
print("\n4. overflow_to_sample_mapping:", inputs["overflow_to_sample_mapping"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Các khóa (keys) trả về: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
Số lượng chunk (cửa sổ trượt) tạo ra: 4
------------------------------------------------------------
1. input_ids (15 token đầu của chunk 0): [101, 2054, 2003, 1996, 2364, 7337, 2109, 2011, 19081, 1029, 102, 1996, 10938, 2121, 4294]
   Giải mã lại text: [CLS] what is the main mechanism used by transformers? [SEP] the transformer architecture

2. attention_mask (15 giá trị đầu của chunk 0): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   Tổng token thật (mask=1) ở chunk 0: 384
   Tổng token thật (mask=1) ở chunk cuối: 280

3. offset_mapping (15 cặp đầu của chunk 0): [(0, 0), (0, 4), (5, 7), (8, 11), (12, 16), (17, 26), (27, 31), (32, 34), (35, 47), (47, 48), (0, 0), (0, 3), (4, 13), (13, 15), (16, 28)]
   Sequence IDs (15 giá trị đầu): [None, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1]

4. overflow_to_sample_mapping: [0, 0, 0, 0]


## Giải thích chi tiết các thành phần trả về

1. **`input_ids`**:
   * Danh sách các chỉ số nguyên (ID) của từng subword token theo từ điển mô hình.
   * Cấu trúc chuỗi: `[CLS] <question_tokens> [SEP] <context_chunk_tokens> [SEP] [PAD]...`. Nhờ `truncation="only_second"`, phần câu hỏi luôn được giữ nguyên, chỉ có `context` bị cắt và chia thành nhiều đoạn dài tối đa `max_length = 384`.

2. **`attention_mask`**:
   * Mảng nhị phân gồm `1` và `0` cùng chiều dài với `input_ids`.
   * Giá trị `1` đánh dấu token thật (bao gồm `[CLS]`, `[SEP]`), còn `0` đánh dấu token đệm (`[PAD]`) ở phần đuôi của chunk cuối cùng để Self-Attention bỏ qua.

3. **`offset_mapping`**:
   * Danh sách cặp tọa độ ký tự `(start_char, end_char)` của từng token trong văn bản gốc.
   * Dùng để ánh xạ nhãn đáp án từ vị trí ký tự (`answer_start` trong SQuAD) sang vị trí token `(start_token, end_token)` khi huấn luyện và giải mã ngược chuỗi trả lời khi suy luận. Các token đặc biệt (`[CLS]`, `[SEP]`, `[PAD]`) có offset là `(0, 0)`.

4. **`overflow_to_sample_mapping`**:
   * Danh sách ánh xạ ngược chỉ số từ mỗi chunk (feature) về chỉ số của mẫu dữ liệu gốc ban đầu (`example_idx`).
   * Vì một cặp `(question, context)` dài bị tách thành nhiều chunk chồng lấp với bước trượt `stride = 128` (ví dụ `[0, 0, 0]` nghĩa là cả 3 chunk đều thuộc mẫu số `0`), trường này giúp chương trình gán đúng nhãn khi train và tổng hợp logit cao nhất trên toàn bộ các chunk của cùng một câu hỏi khi dự đoán.